# Clinical Companion graph

Checks for `app/internal/graph.py`.

The compiled graph starts at `guard`. A rule match sends the turn straight to a specialist. Otherwise the router model chooses a label:

- `DANGEROUS` → `Dangerous` → end, and the question is stored on `issues`
- `INFORMATIONAL` → `plan` → `search` → `grade` → `answer` → end. A `REVISE` grade searches once more, then the answer quotes the ebook and includes BMI when height and weight are present
- `SMALLTALK` → `Small_Talk` → end
- `END` → end

Use the agent virtualenv as the kernel (`FinalProject/app/agent/.venv`). The cells below stub `complete` and `BookIndex.search`, so they do not call Hugging Face or open the ebook. The last cell calls the real model only when `RUN_LIVE` is set.

In [8]:
from pathlib import Path
import sys

def find_agent_root() -> Path:
    seeds = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for seed in seeds:
        if (seed / "app" / "internal" / "graph.py").is_file():
            return seed
        nested = seed / "FinalProject" / "app" / "agent"
        if (nested / "app" / "internal" / "graph.py").is_file():
            return nested
    raise FileNotFoundError(
        "Could not find app/internal/graph.py. "
        "Open this notebook from the repo root or from FinalProject/app/agent."
    )

AGENT_ROOT = find_agent_root()
if str(AGENT_ROOT) not in sys.path:
    sys.path.insert(0, str(AGENT_ROOT))

from dotenv import load_dotenv

load_dotenv(AGENT_ROOT.parent / ".env")

import importlib

import app.internal.book as book_module
import app.internal.graph as graph_module

importlib.reload(book_module)
importlib.reload(graph_module)

from langchain_core.messages import HumanMessage
from app.internal.graph import (
    ANSWER_PROMPT,
    REFUSAL,
    ROUTING_PROMPT,
    SMALL_TALK_PROMPT,
    build_graph,
    companion,
)

AGENT_ROOT

PosixPath('/Users/johnwarrior/Developer/agentic-frameworks/FinalProject/app/agent')

## Graph shape

`build_graph` compiles a checkpointer-backed router. `guard` is the entry node. Informational turns run `plan` → `search` → `grade`, and `grade` can return to `plan` once before `answer`. `Dangerous` ends the turn.

In [9]:
drawn = build_graph().get_graph()
nodes = set(drawn.nodes)

def endpoints(edge):
    source = getattr(edge, "source", None)
    target = getattr(edge, "target", None)
    if source is None:
        source, target = edge[0], edge[1]
    return source, target

edges = {endpoints(edge) for edge in drawn.edges}

expected = {
    ("__start__", "guard"),
    ("guard", "Dangerous"),
    ("guard", "plan"),
    ("guard", "Router"),
    ("Router", "Small_Talk"),
    ("Router", "Dangerous"),
    ("Router", "plan"),
    ("Router", "__end__"),
    ("plan", "search"),
    ("search", "grade"),
    ("grade", "plan"),
    ("grade", "answer"),
    ("answer", "__end__"),
    ("Dangerous", "__end__"),
    ("Small_Talk", "__end__"),
}
missing = expected - edges
assert {"guard", "Router", "Small_Talk", "Dangerous", "plan", "search", "grade", "answer"} <= nodes, sorted(nodes)
assert not missing, sorted(missing)

print("nodes:", sorted(nodes))
print("edges:", sorted(edges))

mermaid = drawn.draw_mermaid()
try:
    from IPython.display import Markdown, display

    display(Markdown(f"```mermaid\n{mermaid}\n```"))
except ImportError:
    print(mermaid)

nodes: ['Dangerous', 'Router', 'Small_Talk', '__end__', '__start__', 'answer', 'grade', 'guard', 'plan', 'search']
edges: [('Dangerous', '__end__'), ('Router', 'Dangerous'), ('Router', 'Small_Talk'), ('Router', '__end__'), ('Router', 'plan'), ('Small_Talk', '__end__'), ('__start__', 'guard'), ('answer', '__end__'), ('grade', 'answer'), ('grade', 'plan'), ('guard', 'Dangerous'), ('guard', 'Router'), ('guard', 'plan'), ('plan', 'search'), ('search', 'grade')]


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	guard(guard)
	Router(Router)
	Small_Talk(Small_Talk)
	Dangerous(Dangerous)
	plan(plan)
	search(search)
	grade(grade)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	Router -. &nbsp;DANGEROUS&nbsp; .-> Dangerous;
	Router -. &nbsp;SMALLTALK&nbsp; .-> Small_Talk;
	Router -. &nbsp;END&nbsp; .-> __end__;
	Router -. &nbsp;INFORMATIONAL&nbsp; .-> plan;
	__start__ --> guard;
	grade -.-> answer;
	grade -.-> plan;
	guard -. &nbsp;DANGEROUS&nbsp; .-> Dangerous;
	guard -. &nbsp;ASK_MODEL&nbsp; .-> Router;
	guard -. &nbsp;INFORMATIONAL&nbsp; .-> plan;
	plan --> search;
	search --> grade;
	Dangerous --> __end__;
	Small_Talk --> __end__;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## `Dangerous`

A diagnosis question matches the guardrail, so the model is not called. The reply is the fixed refusal, and the question is appended to `issues`.

In [5]:
import uuid
from unittest.mock import patch

def turn(text):
    return {
        "messages": [HumanMessage(content=text)],
        "retrieved": "",
        "issues": [],
        "route": "",
        "search_query": "",
        "searches": 0,
        "coverage": "",
    }

def thread():
    return {"configurable": {"thread_id": f"notebook-{uuid.uuid4().hex}"}}

def fail_complete(messages, max_tokens=512):
    raise AssertionError("model should not be called")

with patch("app.internal.graph.complete", fail_complete):
    refused = companion.invoke(turn("Do I have diabetes?"), thread())

assert refused["messages"][-1].content == REFUSAL
assert refused["issues"] == ["Do I have diabetes?"]
assert refused["retrieved"] == ""
print(refused["messages"][-1].content)
print("issues:", refused["issues"])

I can't help with diagnosis or treatment plans. Please contact a clinician or specialist for this question. This is informational guidance only.
issues: ['Do I have diabetes?']


## Informational answer

A diet question plans keywords, searches the ebook, and grades the passages. `SUFFICIENT` goes to the answer, which receives `ANSWER_PROMPT` and the stubbed page passage. `REVISE` searches a second time.

In [7]:
calls = []
queries = []
replies = iter(["breakfast fiber", "SUFFICIENT", "a quoted answer"])

def fake_complete(messages, max_tokens=512):
    calls.append({"messages": messages, "max_tokens": max_tokens})
    return next(replies)

def record_search(self, query):
    queries.append(query)
    return "page 3: fiber at breakfast"

with patch("app.internal.book.BookIndex.search", record_search):
    with patch("app.internal.graph.complete", fake_complete):
        diet = companion.invoke(turn("What is a good diet?"), thread())

sent = calls[-1]["messages"]
assert queries == ["breakfast fiber"]
assert diet["retrieved"] == "page 3: fiber at breakfast"
assert calls[-1]["max_tokens"] == 1024
assert sent[0].content == ANSWER_PROMPT
assert sent[-1].content == "Retrieved context:\npage 3: fiber at breakfast"
assert diet["messages"][-1].content == "a quoted answer"
assert diet["issues"] == []

for message in sent:
    print(f"{message.type}: {message.content[:90]}")
print("reply:", diet["messages"][-1].content)

system: You are Clinical Companion, an informational wellness assistant. Explain the answer in the
human: What is a good diet?
system: Retrieved context:
page 3: fiber at breakfast
reply: a quoted answer


In [8]:
revised = []
revise_replies = iter(
    ["breakfast fiber", "REVISE", "morning meal habits", "SUFFICIENT", "elaborated"]
)

def record_revision(self, query):
    revised.append(query)
    return f"page {len(revised)}: {query}"

with patch("app.internal.book.BookIndex.search", record_revision):
    with patch("app.internal.graph.complete", lambda messages, max_tokens=512: next(revise_replies)):
        second = companion.invoke(turn("What is a good diet?"), thread())

assert revised == ["breakfast fiber", "morning meal habits"]
assert "page 1: breakfast fiber" in second["retrieved"]
assert "page 2: morning meal habits" in second["retrieved"]
assert second["searches"] == 2
assert second["messages"][-1].content == "elaborated"

bmi_replies = iter(["movement habits", "SUFFICIENT", "grounded"])
with patch("app.internal.book.BookIndex.search", return_value="page 4: daily movement"):
    with patch("app.internal.graph.complete", lambda messages, max_tokens=512: next(bmi_replies)):
        measured = companion.invoke(
            turn("I weigh 82 kg and I am 1.78 m. What habits help?"),
            thread(),
        )

assert measured["retrieved"].startswith("BMI 25.9 (overweight)")
assert "page 4: daily movement" in measured["retrieved"]
assert measured["messages"][-1].content == "grounded"
print(second["retrieved"])
print(measured["retrieved"])

page 1: breakfast fiber

page 2: morning meal habits
BMI 25.9 (overweight)

page 4: daily movement


## Model labels

Greetings and unmatched questions skip the guardrail. `complete` must return `SMALLTALK` or `END`. Any other label is treated as `DANGEROUS`. MemorySaver needs a `thread_id`.

In [10]:
smalltalk_calls = []

def smalltalk_complete(messages, max_tokens=512):
    smalltalk_calls.append(messages)
    return "SMALLTALK" if len(smalltalk_calls) == 1 else "Hello from companion."

with patch("app.internal.graph.complete", smalltalk_complete):
    smalltalk = companion.invoke(turn("hello"), thread())

assert smalltalk_calls[0][0].content == ROUTING_PROMPT
assert smalltalk_calls[1][0].content == SMALL_TALK_PROMPT
assert smalltalk["messages"][-1].content == "Hello from companion."
assert smalltalk["retrieved"] == ""

with patch("app.internal.graph.complete", lambda messages, max_tokens=512: "END"):
    ended = companion.invoke(turn("What is the capital of France?"), thread())

assert ended["messages"][-1].content == "END"
assert ended["retrieved"] == ""

with patch("app.internal.graph.complete", lambda messages, max_tokens=512: "maybe"):
    unknown = companion.invoke(turn("What is the capital of France?"), thread())

assert unknown["messages"][-1].content == REFUSAL
assert unknown["issues"] == ["What is the capital of France?"]

print("smalltalk:", smalltalk["messages"][-1].content)
print("end:", ended["messages"][-1].content)
print("unknown:", unknown["messages"][-1].content)

smalltalk: Hello from companion.
end: END
unknown: I can't help with diagnosis or treatment plans. Please contact a clinician or specialist for this question. This is informational guidance only.


## Live model

This calls Hugging Face through smolagents. A greeting should route to `Small_Talk`, so it does not build the PDF index. Set `RUN_LIVE = True` and put an `HF_TOKEN` that starts with `hf_` in `FinalProject/app/.env`.

In [12]:
import os

RUN_LIVE = False

token = (os.getenv("HF_TOKEN") or "").strip()
if not RUN_LIVE:
    print("skipped: set RUN_LIVE = True to call the model")
elif not token.startswith("hf_"):
    print("skipped: set HF_TOKEN in FinalProject/app/.env")
else:
    result = companion.invoke(
        {
            "messages": [HumanMessage(content="Hello")],
            "retrieved": "",
            "issues": [],
            "route": "",
            "search_query": "",
            "searches": 0,
            "coverage": "",
        },
        {"configurable": {"thread_id": f"live-{uuid.uuid4().hex}"}},
    )
    print(result["messages"][-1].content)

skipped: set RUN_LIVE = True to call the model
